# 1. Introdução

O problema de criação de horários académicos é um desafio recorrente em instituições de ensino superior.
A atribuição manual de horários envolve gerir múltiplas variáveis e restrições, como professores, salas, cursos e,
tempos disponíveis, o que aumenta a complexidade e o risco de erros.

Para ajudar a combater este problema, o foco do projeto consiste em criar um **agente inteligente** capaz de criar e gerir de forma atenuada horários, 
Posto isto, iremos solucionar uma solução através do **Problema de Satisfação de Restrições (CSP)**. 
O trabalho será desenvolvido em grupo, recorrendo à linguagem **Python**, ao ambiente **Jupyter Notebook** e à biblioteca **python-constraint**,
de forma a integrar a parte prática de programação com a documentação e explicação do processo.

**Grupo G15**  
- Nuno Miguel Rodrigues da Silva (a25417)  
- Hugo Francisco Pereira Azevedo (a25431)  
- Tiago André Alves Castro (a25434)  
- Gonçalo Edi Pinto Homem de Figueiredo (a26747) 
- João Miguel Santos Silva Campos Cunha (a23519)




# 2. Formulação do Objetivo

O objetivo principal é criar um agente que consiga **administrar horários de aulas** para uma instituição de ensino superior, 
respeitando restrições de natureza distinta:

- **Soluções possíveis:** qualquer atribuição completa, mesmo que viole restrições;
- **Soluções viáveis:** horários que respeitam todas as **restrições hard** obrigatórias;
- **Soluções ótimas:** horários que, além de viáveis, maximizam o cumprimento de **restrições soft**, 
resultando em maior qualidade para alunos e professores;
- **Melhor solução encontrada:** a melhor solução obtida pela implementação, tendo em conta o tempo de execução.




# 3. Planeamento e Desenho do Agente (CSP)

Como já referido, o problema de criar horários pode ser formalizado como um **Problema de Satisfação de Restrições (CSP)**.  

### Variáveis
- **Aulas**: cada disciplina de cada turma origina uma ou duas aulas semanais; 
- **Professores**: associados a disciplinas específicas;
- **Turmas**: grupos de estudantes com um conjunto de disciplinas;
- **Salas**: laboratórios, salas normais ou modalidade online;
- **Slots temporais**: blocos de 2 horas, de segunda a sexta-feira, entre o bloco 1 e 20 (4 blocos x 5 dias).  

### Domínios
- Cada variável *aula* pode ser atribuída a qualquer slot temporal livre.  
- O domínio das aulas é restrito pelos professores disponíveis, pelas salas permitidas e pelas regras definidas no enunciado.  

### Hard Constraints (obrigatórias)
- Todas as aulas têm duração de **2 horas**; 
- Cada turma deve ter **10 aulas por semana**;
- Cada disciplina pode ter **1 ou 2 aulas por semana**;
- Uma turma não pode ter mais de **3 aulas por dia**;
- O horário deve respeitar a **disponibilidade dos professores**;  
- Uma turma não pode ter duas aulas no mesmo horário;
- Um professor não pode dar duas aulas no mesmo horário;  
- Uma sala não pode ser atribuída a duas aulas no mesmo horário.  

### Soft Constraints (preferências)
- As aulas da mesma disciplina devem estar distribuídas em **dias distintos**;
- Cada turma deve, idealmente, ter aulas em **apenas 4 dias por semana**;
- As aulas devem ser **consecutivas** dentro do mesmo dia, reduzindo tempos mortos.  

### Challenges (opcionais)
- As aulas **online**  devem ter apenas 3 alunos por turma e devem ser **marcadas no mesmo dia**;
- Algumas disciplinas se possível, devem ter **sala específica**; 
- O número de salas diferentes usadas por cada turma deve ser **minimizado**.  

### Heurísticas aplicadas
Para otimizar a procura de soluções viáveis e de qualidade, podem ser aplicadas heurísticas como:
- **MRV (Minimum Remaining Values)** – há variáveis muito limitadas, como professores com poucas disponibilidades e,
o MRV dá prioridade a essas variáveis críticas, reduzindo logo o risco de ficar sem solução; 
- **ARC-3** – Reforça a consistência entre variáveis antes de avançar e assim, valores impossíveis são eliminados logo do domínio;
- **LCV (Least Constraining Value)** – escolher horários que deixam mais flexibilidade às restantes turmas/disciplinas.



# 4. Agent Running

Nesta fase, o agente inteligente é implementado de acordo com o *dataset Tiny* fornecido pelo docente (`ClassTT_01_tiny.txt`).

O objetivo é gerar automaticamente horários válidos para as turmas `t01`, `t02` e `t03`, 
respeitando as restrições indicadas no ficheiro, nomeadamente:
- Cada aula tem 2 horas;
- Cada dia possui 4 blocos de 2 horas, totalizando 20 blocos semanais;
- Cada turma tem 5 disciplinas (`UCxx`);
- Professores e salas têm restrições de disponibilidade;
- Algumas aulas são obrigatoriamente online ou em laboratório.

O modelo segue o paradigma de **Problema de Satisfação de Restrições (CSP)**, 
usando a biblioteca `python-constraint` para definir as variáveis, os domínios e as restrições *hard* e *soft*.


###  Execução e procura de soluções
O agente foi implementado com a biblioteca `python-constraint`, aplicando o método `getSolutionIter()` para gerar múltiplas soluções.  
Durante o processo, foram contabilizadas as soluções encontradas e o tempo total de execução.


In [1]:
# Agent Running - Execução final (Tiny Dataset)
from constraint import Problem, AllDifferentConstraint
import time
from collections import defaultdict

# Estrutura do problema
slots = list(range(1, 21))  # 20 blocos (5 dias x 4 blocos)
classes = {
    "t01": ["UC11", "UC12", "UC13", "UC14", "UC15"],
    "t02": ["UC21", "UC22", "UC23", "UC24", "UC25"],
    "t03": ["UC31", "UC32", "UC33", "UC34", "UC35"]
}
teachers = {
    "jo":   ["UC11", "UC21", "UC22", "UC31"],
    "mike": ["UC12", "UC23", "UC32"],
    "rob":  ["UC13", "UC14", "UC24", "UC33"],
    "sue":  ["UC15", "UC25", "UC34", "UC35"]
}
unavailable = {
    "mike": [13,14,15,16,17,18,19,20],
    "rob":  [1,2,3,4],
    "sue":  [9,10,11,12,17,18,19,20]
}
rooms = {"UC14": "Lab01", "UC22": "Lab01"}


# Funções auxiliares 
def day_of(slot): 
    return (slot - 1) // 4

def block_of(slot):
    return (slot - 1) % 4 + 1

def different_day(s1, s2): 
    return day_of(s1) != day_of(s2)

def consecutivos(blocos):
    """Verifica se blocos são consecutivos (ex.: [1,2] ou [2,3,4])."""
    if len(blocos) <= 1:
        return True
    b = sorted(blocos)
    return all(b[i+1] - b[i] == 1 for i in range(len(b)-1))


# Criação do CSP
problem = Problem()
vars_keys = []
for turma, disciplinas in classes.items():
    for disc in disciplinas:
        for lesson in ("lesson1", "lesson2"):
            prof = next(t for t, ds in teachers.items() if disc in ds)
            dominio = [s for s in slots if prof not in unavailable or s not in unavailable[prof]]
            key = (turma, disc, prof, lesson)
            vars_keys.append(key)
            problem.addVariable(key, dominio)


# Restrições HARD

# Professores e turmas não podem coincidir no mesmo slot
for prof in teachers:
    aulas_prof = [v for v in vars_keys if v[2] == prof]
    problem.addConstraint(AllDifferentConstraint(), aulas_prof)
for turma in classes:
    aulas_turma = [v for v in vars_keys if v[0] == turma]
    problem.addConstraint(AllDifferentConstraint(), aulas_turma)
    
# Aulas da mesma UC em dias diferentes
for turma, disciplinas in classes.items():
    for disc in disciplinas:
        v = [k for k in vars_keys if k[0] == turma and k[1] == disc]
        if len(v) == 2: 
            problem.addConstraint(different_day, v)
            
# Lab01 exclusiva (UC14 e UC22)
lab_courses = [c for c, r in rooms.items() if r == "Lab01"]
lab_vars = [v for v in vars_keys if v[1] in lab_courses]
problem.addConstraint(AllDifferentConstraint(), lab_vars)

# Máximo de 3 aulas por dia por turma
def max3_por_dia(*slots_atribuidos):
    for d in range(5):  # 5 dias úteis
        if sum(1 for s in slots_atribuidos if day_of(s) == d) > 3:
            return False
    return True

for turma in classes.keys():
    aulas_turma = [k for k in vars_keys if k[0] == turma]
    problem.addConstraint(max3_por_dia, aulas_turma)


# Execução
print("A executar agente...")
start = time.time()
solutions_iter = problem.getSolutionIter()

solutions = []
for i, sol in enumerate(solutions_iter):
    solutions.append(sol)
    if i >= 100:  # limite prático
        break
end = time.time()

print(f"\nTotal de soluções encontradas: {len(solutions)}")
print(f"Tempo total: {end - start:.3f} segundos")



# Soft Constraints

def avaliar(solucao):
    score = 0
    por_turma_dia = defaultdict(lambda: defaultdict(list))

    # Agrupa blocos por turma e dia
    for (turma, disc, prof, lesson), slot in solucao.items():
        por_turma_dia[turma][day_of(slot)].append(block_of(slot))

    for turma, dias in por_turma_dia.items():
        dias_usados = len(dias)

        #  Preferir até 4 dias de aulas
        if dias_usados <= 4:
            score += 5
        else:
            score -= (dias_usados - 4) * 2

        # Preferir blocos consecutivos por dia
        for blocos in dias.values():
            if consecutivos(blocos):
                score += 2
            else:
                score -= 1

    return score


# Escolher melhor solução
if solutions:
    melhor = max(solutions, key=avaliar)
    melhor_score = avaliar(melhor)

    print(f"\nMelhor solução (score={melhor_score}):\n")
    nomes_dias = ["Seg", "Ter", "Qua", "Qui", "Sex"]
    for aula_key, slot in sorted(melhor.items(), key=lambda kv: kv[1]):
        turma, disc, prof, lesson = aula_key
        dia = nomes_dias[day_of(slot)]
        bloco = (slot - 1) % 4 + 1
        print(f"{dia} - Bloco {bloco:>2} | {turma} {disc} ({lesson}) com {prof}")
else:
    print("\nNenhuma solução encontrada.")


A executar agente...

Total de soluções encontradas: 101
Tempo total: 0.004 segundos

Melhor solução (score=24):

Ter - Bloco  1 | t03 UC35 (lesson2) com sue
Ter - Bloco  2 | t03 UC34 (lesson2) com sue
Ter - Bloco  2 | t02 UC23 (lesson2) com mike
Ter - Bloco  3 | t02 UC25 (lesson2) com sue
Ter - Bloco  3 | t01 UC12 (lesson2) com mike
Ter - Bloco  4 | t01 UC15 (lesson2) com sue
Ter - Bloco  4 | t03 UC32 (lesson2) com mike
Qua - Bloco  2 | t01 UC12 (lesson1) com mike
Qua - Bloco  2 | t02 UC21 (lesson2) com jo
Qua - Bloco  3 | t02 UC23 (lesson1) com mike
Qua - Bloco  3 | t03 UC31 (lesson2) com jo
Qua - Bloco  4 | t03 UC32 (lesson1) com mike
Qua - Bloco  4 | t01 UC11 (lesson2) com jo
Qui - Bloco  1 | t03 UC35 (lesson1) com sue
Qui - Bloco  1 | t02 UC24 (lesson2) com rob
Qui - Bloco  2 | t03 UC34 (lesson1) com sue
Qui - Bloco  2 | t01 UC13 (lesson2) com rob
Qui - Bloco  3 | t02 UC22 (lesson2) com jo
Qui - Bloco  3 | t01 UC15 (lesson1) com sue
Qui - Bloco  3 | t03 UC33 (lesson2) com rob
Qui 

### Análise dos Resultados

O agente inteligente foi capaz de gerar e encontrar **soluções viáveis** com base no ficheiro *Tiny*, 
respeitando as restrições impostas no enunciado:
- Professores não têm sobreposição de horários;
- Turmas têm todas as aulas atribuídas a blocos distintos;
- As restrições de indisponibilidade foram respeitadas;
- O número total de soluções encontradas;
- O máximo de três aulas por dia foi garantido para todas as turmas;
- A sala de laboratório (Lab01) foi corretamente atribuída de forma exclusiva;
- O tempo total de execução;
- O horário correspondente à melhor solução gerada.

O agente, portanto, **encontra e otimiza horários automaticamente**, cumprindo todas as restrições obrigatórias e avaliando as soluções conforme critérios de qualidade.

# 5. Conclusão

O desenvolvimento deste projeto permitiu compreender e aplicar de forma prática os conceitos fundamentais 
da **Inteligência Artificial**, em particular o uso de **Problemas de Satisfação de Restrições (CSP)** 
para a resolução de problemas de planeamento.

Através da biblioteca `python-constraint`, foi possível construir um **agente inteligente** capaz de gerar 
**horários académicos válidos**, respeitando as restrições definidas para professores, turmas, salas e horários.  
Durante o processo de desenvolvimento, foi criado um modelo genérico e escalável, capaz de se adaptar a diferentes 
cenários e conjuntos de dados.

Os resultados obtidos demonstraram que o agente é eficaz a nível computacional e cumpre as **restrições hard**, 
como a não sobreposição de aulas e o respeito pelas disponibilidades dos professores. O agente encontrou diversas soluções viáveis;
onde foi capaz de selecionar automaticamente a **melhor solução**, de acordo com as preferências definidas;

O projeto evidenciou também alguns desafios, nomeadamente:
- A complexidade combinatória do problema, que aumenta exponencialmente com o número de turmas e disciplinas;
- A necessidade de otimizar a procura de soluções através de **heurísticas** (MRV, LCV e AC-3).

### Considerações Finais

Em suma, o agente desenvolvido cumpriu o objetivo proposto: gerar automaticamente horários viáveis 
para diferentes turmas e professores.  
Como trabalho futuro, seria interessante:
- Explorar heurísticas e algoritmos de procura informada para melhorar o desempenho;
- Expandir o modelo para conjuntos de dados mais complexos;
- Desenvolver uma **interface visual** para visualização e exportação dos horários.

O projeto representou uma experiência completa de integração entre **conceitos de IA, raciocínio automático 
e programação prática**, demonstrando o potencial dos CSPs na resolução de problemas reais.
